In [134]:
import pandas as pd

base_path = r"C:/Users/dell/Desktop/CDAC/Eccomerce Sales & Customers Analytics - Mini Project/Data/"

# Load datasets
customers = pd.read_csv(base_path + "olist_customers_dataset.csv")
orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
payments = pd.read_csv(base_path + "olist_order_payments_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(base_path + "olist_products_dataset.csv")


# -------------------------------
# STEP 1: Sample customers
# -------------------------------
customers_small = customers.sample(n=1000, random_state=42)


# -------------------------------
# STEP 2: Filter orders for these customers
# -------------------------------
orders_small = orders[
    orders["customer_id"].isin(customers_small["customer_id"])
]


# -------------------------------
# STEP 3: Re-filter customers (IMPORTANT for integrity)
# Keep only customers who actually have orders
# -------------------------------
valid_customers = orders_small["customer_id"].unique()

customers_small = customers_small[
    customers_small["customer_id"].isin(valid_customers)
]


# -------------------------------
# STEP 4: Rebuild orders after cleaning customers
# -------------------------------
orders_small = orders[
    orders["customer_id"].isin(customers_small["customer_id"])
]


# -------------------------------
# STEP 5: Payments linked to orders
# -------------------------------
payments_small = payments[
    payments["order_id"].isin(orders_small["order_id"])
]


# -------------------------------
# STEP 6: Order Items (VERY IMPORTANT)
# -------------------------------
order_items_small = order_items[
    order_items["order_id"].isin(orders_small["order_id"])
]


# -------------------------------
# STEP 7: Reviews (VERY IMPORTANT)
# -------------------------------
reviews_small = reviews[
    reviews["order_id"].isin(orders_small["order_id"])
]

products_small = products[
    products["product_id"].isin(order_items_small["product_id"])
]

# -------------------------------
# FINAL SHAPE CHECK
# -------------------------------
print("Customers:", customers_small.shape)
print("Orders:", orders_small.shape)
print("Payments:", payments_small.shape)
print("Order Items:", order_items_small.shape)
print("Reviews:", reviews_small.shape)
print("Products:",products_small.shape)

Customers: (1000, 5)
Orders: (1000, 8)
Payments: (1049, 5)
Order Items: (1130, 7)
Reviews: (996, 7)
Products: (923, 9)


In [136]:
from sqlalchemy import create_engine

user = "root"
password = "Sakshi%4029"   # encoded @ → %40
host = "localhost"
db = "ecommerce_analysis"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:3306/{db}",
    pool_pre_ping=True
)

In [138]:
customers_small.to_sql("customers", engine, if_exists="replace", index=False)


1000

In [140]:
orders_small.to_sql("orders", engine, if_exists="replace", index=False)

1000

In [142]:
payments_small.to_sql("payments", engine, if_exists="replace", index=False)

1049

In [144]:
order_items_small.to_sql("order_items", engine, if_exists="replace", index=False)

1130

In [146]:
reviews_small.to_sql("reviews", engine, if_exists="replace", index=False)

996

In [148]:
products_small.to_sql(
    "products",
    engine,
    if_exists="replace",
    index=False
)

923

In [150]:
pd.read_sql("SHOW TABLES", engine)

,Tables_in_ecommerce_analysis
0,customers
1,order_items
2,orders
3,payments
4,products
5,reviews


In [152]:
#Total Revenue
pd.read_sql("""
SELECT SUM(payment_value) AS total_revenue
FROM payments;
""", engine)

,total_revenue
0,169405.44


In [154]:
#Total Orders
pd.read_sql("""
SELECT COUNT(DISTINCT order_id) AS total_orders
FROM orders;
""", engine)

,total_orders
0,1000


In [156]:
#Total Customers
pd.read_sql("""
SELECT COUNT(*) AS total_customers
FROM customers;
""", engine)

,total_customers
0,1000


In [158]:
pd.read_sql("""
DESCRIBE orders;
""", engine)

,Field,Type,Null,Key,Default,Extra
0,order_id,text,YES,,None,
1,customer_id,text,YES,,None,
2,order_status,text,YES,,None,
3,order_purchase_timestamp,text,YES,,None,
4,order_approved_at,text,YES,,None,
5,order_delivered_carrier_date,text,YES,,None,
6,order_delivered_customer_date,text,YES,,None,
7,order_estimated_delivery_date,text,YES,,None,


In [160]:
orders_small["order_purchase_timestamp"] = pd.to_datetime(
    orders_small["order_purchase_timestamp"]
)

C:\Users\dell\AppData\Local\Temp\ipykernel_4288\2979724827.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_small["order_purchase_timestamp"] = pd.to_datetime(


In [162]:
orders_small.to_sql(
    "orders",
    engine,
    if_exists="replace",
    index=False
)

1000

In [164]:
pd.read_sql("DESCRIBE orders", engine)


,Field,Type,Null,Key,Default,Extra
0,order_id,text,YES,,None,
1,customer_id,text,YES,,None,
2,order_status,text,YES,,None,
3,order_purchase_timestamp,datetime,YES,,None,
4,order_approved_at,text,YES,,None,
5,order_delivered_carrier_date,text,YES,,None,
6,order_delivered_customer_date,text,YES,,None,
7,order_estimated_delivery_date,text,YES,,None,


In [166]:
# Monthly Sales Trend
import pandas as pd

pd.read_sql("""
SELECT 
    DATE_FORMAT(order_purchase_timestamp, '%%Y-%%m') AS month,
    COUNT(order_id) AS total_orders
FROM orders
GROUP BY DATE_FORMAT(order_purchase_timestamp, '%%Y-%%m')
ORDER BY month;
""", engine)

,month,total_orders
0,2016-10,1
1,2017-01,12
2,2017-02,25
3,2017-03,29
4,2017-04,23
5,2017-05,31
6,2017-06,32
7,2017-07,27
8,2017-08,44
9,2017-09,41


In [168]:
#Top 10 Customer Spending
pd.read_sql("""
SELECT 
    c.customer_id,
    SUM(p.payment_value) AS total_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN payments p ON o.order_id = p.order_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 10;
""", engine)

,customer_id,total_spent
0,0b16ce67087d02bf833c807e82b1992b,3358.24
1,6e568303db60ca2bb3aec0b1f9183492,2234.70
2,0ef4273145d74465c180b6a9fbb9799b,1898.61
3,ee517d3a38e003aabc2dc2b08a0bc6fb,1843.85
4,9ac5cdcba84c5425f84b159685728624,1648.64
5,4fc9e35cc1c1237488346652e2487231,1575.05
6,250dfa98c84cdde19347af0428388a8c,1429.59
7,cc9e69978aa877c0442093a193254a67,1367.23
8,98b57da4f6372460e9bd54e2e4491e0b,1337.19
9,d99a2c8000e14e11c6cf41a2352a9e91,1322.69


In [170]:
# Monthly Revenue Trend
pd.read_sql("""
SELECT 
    SUBSTRING(o.order_purchase_timestamp, 1, 7) AS month,
    ROUND(SUM(p.payment_value), 2) AS revenue
FROM orders o
JOIN payments p ON o.order_id = p.order_id
GROUP BY SUBSTRING(o.order_purchase_timestamp, 1, 7)
ORDER BY month;
""", engine)

,month,revenue
0,2016-10,406.92
1,2017-01,2790.19
2,2017-02,4013.88
3,2017-03,5472.78
4,2017-04,4617.16
5,2017-05,4835.99
6,2017-06,3921.08
7,2017-07,4635.92
8,2017-08,6382.39
9,2017-09,6929.97


In [172]:
# MONTH-OVER-MONTH (MoM) GROWTH %
pd.read_sql("""
WITH monthly_revenue AS (
    SELECT 
        SUBSTRING(o.order_purchase_timestamp, 1, 7) AS month,
        SUM(p.payment_value) AS revenue
    FROM orders o
    JOIN payments p ON o.order_id = p.order_id
    GROUP BY SUBSTRING(o.order_purchase_timestamp, 1, 7)
)
SELECT 
    month,
    revenue,
    LAG(revenue) OVER (ORDER BY month) AS prev_month,
    ROUND(
        ((revenue - LAG(revenue) OVER (ORDER BY month)) /
        LAG(revenue) OVER (ORDER BY month)) * 100, 2
    ) AS mom_growth_percent
FROM monthly_revenue;
""", engine)

,month,revenue,prev_month,mom_growth_percent
0,2016-10,406.92,NaN,NaN
1,2017-01,2790.19,406.92,585.69
2,2017-02,4013.88,2790.19,43.86
3,2017-03,5472.78,4013.88,36.35
4,2017-04,4617.16,5472.78,-15.63
5,2017-05,4835.99,4617.16,4.74
6,2017-06,3921.08,4835.99,-18.92
7,2017-07,4635.92,3921.08,18.23
8,2017-08,6382.39,4635.92,37.67
9,2017-09,6929.97,6382.39,8.58


In [174]:
# Top Products Categories
pd.read_sql("""
SELECT 
    p.product_category_name,
    ROUND(SUM(pay.payment_value), 2) AS revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN payments pay ON oi.order_id = pay.order_id
GROUP BY p.product_category_name
ORDER BY revenue DESC
LIMIT 10;
""", engine)

,product_category_name,revenue
0,cama_mesa_banho,19395.65
1,beleza_saude,16843.30
2,utilidades_domesticas,14670.07
3,esporte_lazer,14019.57
4,informatica_acessorios,13845.59
5,moveis_decoracao,13562.91
6,relogios_presentes,13152.70
7,automotivo,10716.14
8,moveis_escritorio,9281.93
9,cool_stuff,7690.53


In [178]:
# Top Sellers by Revenue
pd.read_sql("""
SELECT 
    seller_id,
    ROUND(SUM(price), 2) AS total_sales
FROM order_items
GROUP BY seller_id
ORDER BY total_sales DESC
LIMIT 10;
""", engine)

,seller_id,total_sales
0,7e93a43ef30c4f03f38b393420bc753a,6607.96
1,b37c4c02bda3161a7546a4e6d222d5b2,3475.00
2,955fee9216a65b617aa5c0531780ce60,2549.29
3,7c67e1448b00f6e969d365cea6b010ab,2395.57
4,4869f7a5dfa277a7dca6462dcf3b52b2,2169.60
5,fa1c13f2614d7b5c4749cbc52fecda94,2038.40
6,4a3ca9315b744ce9f8e9374361493884,1844.30
7,e59aa562b9f8076dd550fcddf0e73491,1802.50
8,ba90964cff9b9e0e6f32b23b82465f7b,1799.00
9,40db9e9aa57f7bb151bcda6b0f9bdbb7,1788.00


In [180]:
# Average Order Value
pd.read_sql("""
SELECT 
    ROUND(SUM(payment_value) / COUNT(DISTINCT order_id), 2) AS avg_order_value
FROM payments;
""", engine)

,avg_order_value
0,169.41
